# Ray Unit 0 - Actors
Ray **actors** handle cluster-wide **state**: mutable data that persists across calls.


## Setup

Use the `22971-ray` environment and kernel.



In [1]:
import time
import ray

In [2]:
ray.shutdown()
ray.init()

2026-03-23 10:54:06,104	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
c:\Users\User\.conda\envs\22971-ray\Lib\site-packages\ray\_private\worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.13
Ray version:,2.54.0
Dashboard:,http://127.0.0.1:8265


## 1. Define an actor

Actors are long-lived **class instances**: 
- live on a dedicated process from creation until explicit termination
- maintain mutable state

Define the class:

In [21]:
@ray.remote
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1
        return self.value

    def get_value(self):
        return self.value


Create a class instance (actor):

In [53]:
counter = Counter.remote()

print(counter)
print(type(counter))

Actor(Counter, 580d1e6c0688f2f4d4c4c8c301000000)
<class 'ray.actor.ActorHandle'>


- The counter actor is instantiated somewhere in the cluster.
- The return value is a **handle**: a reference to that instance.


## 2. Call actor methods

Actor methods are invoked with `.remote(...)`, just like task calls.

The method call returns an `ObjectRef`.
Use `ray.get(...)` to retrieve the actual result.

In [24]:
val_ref1 = counter.get_value.remote()
print(val_ref1)
print(ray.get(val_ref1))

counter.increment.remote()

val_ref2 = counter.get_value.remote()
print(val_ref2)
print(ray.get(val_ref2))

ObjectRef(88543757a8df6d2ffd200c7f95163f4059929cdb0100000001000000)
1
ObjectRef(a02c24b8b7fc0a31fd200c7f95163f4059929cdb0100000001000000)
2


## 3. State persists across method calls

Run the previous cell a few times.

**Under the hood:**
- The counter actor is alive throughout and maintains the attribute `self.value`.
- Each `counter.increment.remote()` call updates this attribute.
- Later calls use the updated value.

## 4. Issues with shared mutable state

Sharing state between concurrent workers is difficult:

- **Stale reads**
  - worker #1 reads state `x = 5`
  - worker #2 updates `x → 10`
  - worker #1 continues computation using `x = 5`
  - worker #1's return value is based on **outdated state**

- **Lost update**
  - initial state: `x = 0`
  - worker #1 reads `x = 0`, computes `x + 1`
  - worker #2 reads `x = 0`, computes `x + 1`
  - worker #1 writes `x = 1`
  - worker #2 writes `x = 1`
  - final state: `x = 1` (**one update lost**)

- **Race conditions**
  - initial state: `x = 1`
  - worker #1 reads `x`, computes `x * 2`
  - worker #2 reads `x`, computes `x + 3`
  - depending on execution order:
    - worker #1 → worker #2:
      - worker #1: `x = 1 → 2`
      - worker #2 reads updated `x = 2`, writes `x = 5`
    - worker #2 → worker #1:
      - worker #2: `x = 1 → 4`
      - worker #1 reads updated `x = 4`, writes `x = 8`
  - final state is `5` or `8` → **depends on timing (non-deterministic)**

- ... 
- [(Martin Kleppmann's lectures)](https://www.youtube.com/playlist?list=PLeKd45zvjcDFUEv_ohr_HdUFe97RItdiB)

### Ray's solution:

- Actors **own the state** (no shared mutable state across workers)
- Only the actor can **mutate its own state** (single writer)
- Tasks submit method calls to the actor
- Incoming calls are executed in a **dedicated process**, one at a time, in order of arrival

Result: **deterministic state transitions**

**Caution:** actors serialize execution, not submission—ordering still depends on timing.

In [25]:
@ray.remote
class Sleeper:
    def work(self, label, delay_s):
        start = time.perf_counter()
        print(f"{label}: start")
        time.sleep(delay_s)
        elapsed = time.perf_counter() - start
        print(f"{label}: finished after {elapsed:.2f}s")
        return label

In [29]:
sleeper = Sleeper.remote()

start = time.perf_counter()
refs = [
    sleeper.work.remote("job-1", 2.0),
    sleeper.work.remote("job-2", 2.0),
]
out = ray.get(refs)
elapsed = time.perf_counter() - start

time.sleep(0.1)
print("Output :", out)
print(f"Elapsed: {elapsed:.2f} seconds")

(Sleeper pid=36520) job-1: start
(Sleeper pid=36520) job-1: finished after 2.00s
(Sleeper pid=36520) job-2: start
(Sleeper pid=36520) job-2: finished after 2.00s
Output : ['job-1', 'job-2']
Elapsed: 4.16 seconds


Serial execution `=>` total time is about **4 seconds**, not 2.


## 5. Different actors: independent state, parallel execution


In [36]:
@ray.remote
class Sleeper:
    def __init__(self):
        self.count = 0

    def work(self, name, t):
        self.count += 1
        print(f"[{name}] start")
        time.sleep(t)
        print(f"[{name}] end")
        return self.count


sleeper_a = Sleeper.remote()
sleeper_b = Sleeper.remote()

start = time.perf_counter()
refs = [
    sleeper_a.work.remote("actor-A", 2.0),
    sleeper_b.work.remote("actor-B", 2.0),
]
out = ray.get(refs)
elapsed = time.perf_counter() - start

time.sleep(0.1)  
print("Returned:", out)
print(f"Elapsed: {elapsed:.2f} seconds")

(Sleeper pid=39900) [actor-A] start
(Sleeper pid=36500) [actor-B] start
(Sleeper pid=39900) [actor-A] end
(Sleeper pid=36500) [actor-B] end
Returned: [1, 1]
Elapsed: 2.83 seconds


- Methods on **different** actors run in parallel.
- Each actor has an independent `count` state.

## 6. Actor use case: expensive initialization

Loading a ML model is expensive (I/O, deserialization, warmup), but inference is cheap.

Actors let us **load once and reuse**:
- model is initialized once per actor
- repeated `predict` calls reuse the same in-memory model
- avoids reloading the model on every call

In [37]:
@ray.remote
class ToyModelServer:
    def __init__(self):
        print("Loading model once...")
        time.sleep(2.0)
        self.bias = 0.5

    def predict(self, x):
        return x + self.bias

In [47]:
start = time.perf_counter()
server = ToyModelServer.remote()
created = time.perf_counter() - start

print(f"Actor creation call returned in {created:.2f} seconds")

start = time.perf_counter()
preds = ray.get([server.predict.remote(x) for x in [1.0, 2.0, 3.0]])
elapsed = time.perf_counter() - start

print("Predictions:", preds)
print(f"Time to first predictions: {elapsed:.2f} seconds")

Actor creation call returned in 0.00 seconds
(ToyModelServer pid=2304) Loading model once...
Predictions: [1.5, 2.5, 3.5]
Time to first predictions: 2.70 seconds


Actor creation returns quickly because:

1. `.remote()` submits the actor creation request and immediately returns a handle.
2. Ray schedules the actor on a node.
3. The actor process starts and runs `__init__`.
4. Actor method calls will queue until initialization completes.


The model-loading cost is paid **once per actor**.

After that, later method calls reuse the actor's state (fast predictions).


In [49]:
start = time.perf_counter()
preds = ray.get([server.predict.remote(x) for x in [1.0, 2.0, 3.0]])
elapsed = time.perf_counter() - start

print("Predictions:", preds)
print(f"Elapsed: {elapsed:.2f} seconds")

Predictions: [1.5, 2.5, 3.5]
Elapsed: 0.01 seconds


## 7. Passing actor handles
Tasks can receive actor handles as arguments.

This lets one part of the system talk to a long-lived service owned elsewhere.


In [50]:
@ray.remote
def query_counter(counter_handle, n):
    refs = [counter_handle.increment.remote() for _ in range(n)]
    return ray.get(refs)

In [51]:
counter3 = Counter.remote()
result = ray.get(query_counter.remote(counter3, 3))
final_value = ray.get(counter3.get_value.remote())

print("Task saw   :", result)
print("Actor holds:", final_value)

Task saw   : [1, 2, 3]
Actor holds: 3


The task did not receive the actor's state directly.
It received a **handle** that lets it submit methods to that actor.


## 8. Summary

1. An **actor** is a long-lived remote class instance.
2. Actor methods are invoked with `.remote(...)`.
3. Actor methods return `ObjectRef`s, just like tasks.
4. Methods on the **same** actor run **serially**.
5. Methods on **different** actors can run in parallel.
6. Actors are useful when you need **state**, **caching**, or **expensive initialization**.


## 9. Optional exercises

1. Use the `Counter` class to create an example where stale reads occur.
   - Hint: read state, then delay before using it.
2. Modify `Counter` to introduce **lost updates**:
   - implement `increment(x)` as `state = x + 1`
   - call it with stale values
3. Modify `ToyModelServer` so that it also stores:
   - `num_requests`
   - `last_input`

   Then expose a `stats(...)` method.
4. Create **four** `ToyModelServer` actors and distribute requests across them.
   - Compare one actor vs four actors.
   - What changes in latency and throughput?
   - Hint: route each request to a random actor (e.g. `np.random.randint(4)`)


In [ ]:
ray.shutdown()